# CapGTA_dev — baseline exploration + sweep results

1. Look at the local-baseline distributions (intron vs flanking).
2. Load sweep outputs.
3. Compare predicted RNA-count distributions across configs (and against the spliced-only baseline).

In [ ]:
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_context('notebook')
sns.set_style('whitegrid')

REPO = Path('/fh/fast/srivatsan_s/grp/SrivatsanLab/Dustin/SPC_genome')
DEV = REPO / 'CapGTA_dev'

CAL_PATH  = DEV / 'results' / 'worm6_final' / 'calibration.h5ad'
SWEEP_DIR = DEV / 'results' / 'worm6_final' / 'sweep'

## 1. Baseline distributions

In [ ]:
cal = ad.read_h5ad(CAL_PATH)
print(cal)

def dense(m):
    return m.toarray() if hasattr(m, 'toarray') else np.asarray(m)

intron_bp = cal.var['intron_bp'].to_numpy()
flank_bp  = cal.var['flanking_bp'].to_numpy()
intron_ct = dense(cal.layers['intron']).astype(np.float32)
flank_ct  = dense(cal.layers['flank']).astype(np.float32)

intron_rate = np.where(intron_bp[None] > 0, intron_ct / np.maximum(intron_bp[None], 1), np.nan)
flank_rate  = np.where(flank_bp[None]  > 0, flank_ct  / np.maximum(flank_bp[None], 1),  np.nan)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].hist(np.log10(intron_rate[np.isfinite(intron_rate) & (intron_rate > 0)]),
           bins=80, alpha=0.7, label='intron')
ax[0].hist(np.log10(flank_rate[np.isfinite(flank_rate) & (flank_rate > 0)]),
           bins=80, alpha=0.7, label='flanking')
ax[0].set_xlabel('log10 rate (reads / bp)'); ax[0].set_ylabel('cell x gene entries')
ax[0].legend(); ax[0].set_title('per-locus gDNA rate distribution')

per_gene_intron = np.nanmean(intron_rate, axis=0)
per_gene_flank  = np.nanmean(flank_rate,  axis=0)
ok = np.isfinite(per_gene_intron) & np.isfinite(per_gene_flank)
ax[1].scatter(per_gene_intron[ok], per_gene_flank[ok], s=3, alpha=0.2)
lo, hi = 1e-7, max(np.nanmax(per_gene_intron), np.nanmax(per_gene_flank))
ax[1].plot([lo, hi], [lo, hi], 'r--', lw=1)
ax[1].set_xscale('log'); ax[1].set_yscale('log')
ax[1].set_xlabel('intron rate (mean over cells)'); ax[1].set_ylabel('flanking rate (mean over cells)')
ax[1].set_title('per-gene agreement')
plt.tight_layout()

## 2. Sweep — held-out metrics

In [ ]:
rows = []
for cfg_dir in sorted(SWEEP_DIR.glob('*')):
    metrics = json.loads((cfg_dir / 'metrics.json').read_text())
    config  = json.loads((cfg_dir / 'config.json').read_text())
    rows.append({**config, **metrics, 'tag': cfg_dir.name})
results = pd.DataFrame(rows).sort_values('pearson_log1p', ascending=False)
results

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(results, x='tag', y='pearson_log1p', ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_title('held-out Pearson (log1p) — junction genes')
plt.tight_layout()

## 3. Predicted RNA-count distributions per config

In [ ]:
preds = {}
for cfg_dir in sorted(SWEEP_DIR.glob('*')):
    preds[cfg_dir.name] = ad.read_h5ad(cfg_dir / 'predictions.h5ad')

fig, ax = plt.subplots(figsize=(10, 4))
for tag, p in preds.items():
    per_cell = np.asarray(dense(p.X).sum(axis=1)).ravel()
    ax.hist(np.log10(per_cell + 1), bins=60, alpha=0.3, label=tag)
spliced_per_cell = np.asarray(dense(cal.layers['spliced']).sum(axis=1)).ravel()
ax.hist(np.log10(spliced_per_cell + 1), bins=60, histtype='step',
        color='k', lw=2, label='spliced_only')
ax.set_xlabel('log10( total RNA counts per cell + 1 )'); ax.set_ylabel('cells')
ax.legend(fontsize=7, loc='upper left')
plt.tight_layout()

## 4. Junction vs junction-less: where do models differ?

In [ ]:
n_junc = cal.var['n_junc'].to_numpy()
juncmask = n_junc > 0

fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for tag, p in preds.items():
    per_gene = np.asarray(dense(p.X).mean(axis=0)).ravel()
    ax[0].hist(np.log10(per_gene[~juncmask] + 1), bins=60, alpha=0.35, label=tag)
    ax[1].hist(np.log10(per_gene[juncmask]  + 1), bins=60, alpha=0.35, label=tag)
ax[0].set_title('junction-less genes'); ax[1].set_title('junction-containing')
for a in ax:
    a.set_xlabel('log10( mean prediction per gene + 1 )')
    a.legend(fontsize=7)
plt.tight_layout()